
## Práctica 1: Exploración de Niveles del lenguaje 🔭

### FECHA DE ENTREGA: 10 de Marzo 2026 at 11:59pm



### Fonética

1. Con base en el sistema de búsqueda visto en la práctica 1, dónde se recibe una palabra ortográfica y devuelve sus transcripciones fonológicas, proponga una solución para los casos en que la palabra buscada no se encuentra en el lexicón/diccionario.
    - ¿Cómo devolver o **aproximar** su transcripción fonológica?
    - Reutiliza el sistema de búsqueda visto en clase y mejóralo con esta funcionalidad.
    - Muestra al menos tres ejemplos


Reciclando el procedimiento de la práctica 1, importamos algunos modulos necesarios y obtenemos el corpus.


In [10]:
import http

from collections import defaultdict
import requests as r

import pandas as pd

from rich import print as rprint
from rich.columns import Columns
from rich.panel import Panel
from rich.text import Text

Tomamos los datos desde open-dict.

In [11]:
IPA_URL = "https://raw.githubusercontent.com/open-dict-data/ipa-dict/master/data/{lang}.txt"


Definimos nuestros codigos de lenguaje para obtener el dataset. Seleccionamos solo algunos, en especifico los que vamos a usar en el segundo punto.

In [12]:
lang_codes = {
    # Lenguas Indo-europeas
    "es_ES": "Spanish (Spain)",
    "es_MX": "Spanish (Mexico)",
    # Lengua Japónica
    "ja": "Japanese",
    #Lengua austronesia
    "ma": "Malay (Malaysian and Indonesian)",
    # Algunas lenguas indo-europeas adicionales para el análisis.
    "en_UK": "English (Received Pronunciation)",
    "en_US": "English (General American)",
    "fr_FR": "French (France)",
    "fr_QC": "French (Québec)",
    }


Reutilizamos el codigo para obtener el dataset y descargamos:


In [13]:
def download_ipa_corpus(iso_lang: str) -> str:
    """Get ipa-dict file from Github

    Parameters:
    -----------
    iso_lang:
        Language as iso code

    Results:
    --------
    dict:
        Dictionary with words as keys and phonetic representation
        as values for a given lang code
    """
    print(f"Downloading {iso_lang}", end="::")
    response = r.get(IPA_URL.format(lang=iso_lang))
    status_code = response.status_code
    print(f"status={status_code}")
    if status_code != http.HTTPStatus.OK:
        print(f"ERROR on {iso_lang} :(")
        return ""
    return response.text

In [14]:
def parse_response(response: str) -> dict:
    """Parse text response from ipa-dict to python dict

    Each row have the format:
    [WORD][TAB]/[IPA]/(, /[IPA]/)?

    Parameters
    ----------
    response: str
        ipa-dict raw text

    Returns
    -------
    dict:
        A dictionary with the word as key and the phonetic
        representations as value
    """
    ipa_list = response.rstrip().split("\n")
    result = {}
    for item in ipa_list:
        if item == '':
            continue
        item_list = item.split("\t")
        result[item_list[0]] = item_list[1]
    return result

In [20]:
def get_corpora() -> dict:
    """Download corpora from ipa-dict github

    Given a list of iso lang codes download available datasets.

    Returns
    -------
    dict
        Lang codes as keys and dictionary with words-transcriptions
        as values
    """
    return {
        code: parse_response(download_ipa_corpus(code))
         for code in iso_lang_codes
    }

In [21]:
ipa_corpora = get_corpora()

NameError: name 'iso_lang_codes' is not defined

Teníamos entonces el siguiente sistema naive para devolver la transcripción fonologica a partir de una palabra ortogáfica.

In [18]:
def get_formated_string(code: str, name: str):
    return f"[b]{name}[/b]\n[yellow]{code}"

In [ ]:
rprint(
    Panel(Text("Representación fonética de palabras", style="bold", justify="center"))
)
rendable_langs = [
    Panel(get_formated_string(code, lang), expand=True)
    for code, lang in lang_codes.items()
]
rprint(Columns(rendable_langs))

lang = input("lang>> ")
rprint(f"Selected language: {lang_codes[lang]}") if lang else rprint("Adios 👋🏼")
while lang:
    dataset = corpora[lang]
    query = input(f"  [{lang}]word>> ")
    results = get_ipa_transcriptions(query, dataset)
    print(query, " | ", ", ".join(results))
    while query:
        query = input(f"  [{lang}]word>> ")
        if query:
            results = get_ipa_transcriptions(query, dataset)
            rprint(query, " | ", ", ".join(results))
    lang = input("lang>> ")
    rprint(f"Selected language: [yellow]{lang_codes[lang]}[/]") if lang else rprint(
        "Adios 👋🏼"
    )

Sin embargo, esta implementación fallaba cuando pediamos la representación fonetica de palabras que no estuvieran el el corpus. Esto tiene dos causas probables:




El corpus no contiene a la transcripción  de la que buscamos ni ninguna similar.

Por ejemplo, el sistema falla con palabras de otros idiomas(prestamo lingüistico) que han pasado por un proceso de hibridación, entonces el sistema para el lenguaje español es incapaz de encotrar palabras como _parsear_,_twittear_,_postear_,etc.

En primera estancia este problema es complicado de resolver, pues el sistema busca exactamente la palabra que ingresamos y por el fenómeno del prestamo lingüistico no los encuentra, por lo tanto, pensaría en buscar un corpus que haya contemplado este fenómeno para agregarlo a nuestro dataset. Otra opción, pensando en el caso del español-inglés es implementar algún detector de idioma que detecte que la palabra ingresada es una combinación de una raiz extranjera y alguna estructura del español, de esta manera por ejemplo detectaría que _parsear_ se compone de _parse_ y la terminación del verbo en infinitivo _-ar_ y con ello haría la combinación de la transcripción lingüistica de parse en inglés y la terminación del infinitivo del español.

In [ ]:
# Biblioteca para expresiones regulares.
import re
def transcribir_hibrido_verbo_derivado_en_es(word: str) -> str:
    
    """
    Transcribe un verbo derivado del Ingles al Español a su forma híbrida en IPA.
    Por ejemplo, "parsear" se transcribiría como "/ˈpɑɹs/ 
    
    Parameters:    -----------
    word: str
        Verbo en infinitivo en español que se desea transcribir.
    Returns:
    --------
    str
        La transcipción fonológica híbrida del verbo, o None si no es un verbo derivado definido por la regex o la raiz no está en el corpus.
        """
    #Detecta que sea un verbo en infinitivo y devuelve su raíz.
    regex = r"^(.*?)(ar|er|ir|ando|ado)$"
    match = re.match(regex, word)
   

    if match:
        raiz = match.group(1)
        sufijo = match.group(2)
    
    dataset = corpora[lang]
    raiz_en = get_ipa_transcriptions(raiz, dataset)

    if raiz_en:
        raiz_en = raiz_en[0]
        if sufijo == "ar":
            return raiz_en + "ɐɾ"
        elif sufijo == "er":
            return raiz_en + "eɾ"
        elif sufijo == "ir":
            return raiz_en + "iɾ"
        elif sufijo =="ando":
            return raiz_en + "ɐndo"
        elif sufijo == "ado":
            return raiz_en + "aðo"
        elif sufijo == "":
            return raiz_en
    return None

    
    

In [ ]:
rprint(
      Panel(Text("Representación fonética de palabras", style="bold", justify="center"))
  )
rendable_langs = [
      Panel(get_formated_string(code, lang), expand=True)
      for code, lang in lang_codes.items()
  ]
rprint(Columns(rendable_langs))

lang = input("lang>> ")
rprint(f"Selected language: {lang_codes[lang]}") if lang else rprint("Adios 👋🏼")
while lang:
    dataset = corpora[lang]
    while True:
        query = input(f"  [{lang}]word>> ").strip()        
        if not query:
            break

        # 1. Intentamos buscar el dato en el corpus primero
        results = get_ipa_transcriptions(query, dataset)
        
        # Verificamos que el resultado sea válido y no esté vacío
        if results and results != ['']:
            rprint(f"[bold white]{query}[/] | [green]{', '.join(results)}[/] [italic](Corpus)[/]")
        
        else:
            # 2. En otro caso (si no está en el corpus), hacemos el análisis de palabra híbrida
            hybrid_vb = transcribir_hibrido_verbo_derivado_en_es(query)
            
            if hybrid_vb:
                rprint(f"[bold white]{query}[/] | [cyan]{hybrid_vb}[/] [italic](Análisis Híbrido)[/]")
            else:
                # 3. Si tampoco es híbrida, imprimimos el resultado vacio.
                continue
    lang = input("lang>> ").strip()
    if lang:
        rprint(f"Selected language: [yellow]{lang_codes.get(lang, lang)}[/]")
    else:
        rprint("Adios 👋🏼")

De cualquier manera, vemos que mi propuesta es un sistema bastante limitado, al estar basado en reglas muy basicas y solo tomar en cuenta los casos de anglisismos en el español, el programa es muy limitado. Es posible hacer un programa más complejo mediante aprendizaje de maquina en el que aprenda a leer estos fenómenos, sin embargo, creo que es muy complejo para el análisis de ésta práctica.

La palabra tiene algún error ortográfico.

Como la palabra está mal escrita es imposible que el sistema la encuentre, sin embargo, es posible dar una sugerencia de palabra que pueda ser similar a la que escribimos pero que si esté en el corpus, para ello, buscamos alguna palabra cuya distancia de Levenshtein, buscando en el corpus las palabras con la menor distancia de Levenshtein con respecto a la palabra que escribimos.


In [ ]:
from Levenshtein import distance as l_distance


In [ ]:
def dar_sugerencia_Levenshtein(word: str, dataset: dict, n: int = 5) -> [str]:
    """Dada una palabra y un dataset, devuelve la palabra más cercana
    según la distancia de Levenshtein.

    Parameters
    ----------
    word: str
        Palabra a comparar
    dataset: dict
        Dataset con palabras como claves

    Returns
    -------
    str
        La palabra más cercana en el dataset según Levenshtein.
    """
    return sorted(dataset.keys(), key=lambda x: l_distance(word, x))[:n]


De ésta manera, adjuntando esta solución al codigo de la clase tenemos que:


In [17]:
rprint(
      Panel(Text("Representación fonética de palabras", style="bold", justify="center"))
  )
rendable_langs = [
      Panel(get_formated_string(code, lang), expand=True)
      for code, lang in lang_codes.items()
  ]
rprint(Columns(rendable_langs))

lang = input("lang>> ")
rprint(f"Selected language: {lang_codes[lang]}") if lang else rprint("Adios 👋🏼")
while lang:
    dataset = corpora[lang]
    while True:
        query = input(f"  [{lang}]word>> ").strip()        
        if not query:
            break

        # 1. Intentamos buscar el dato en el corpus primero
        results = get_ipa_transcriptions(query, dataset)
        
        # Verificamos que el resultado sea válido y no esté vacío
        if results and results != ['']:
            rprint(f"[bold white]{query}[/] | [green]{', '.join(results)}[/] [italic](Corpus)[/]")
        
        else:
            # 3. Vemos si nos equivocamos ortograficamente y damos sugerencias.
            suggestions = dar_sugerencia_Levenshtein(query, dataset)
            if suggestions:
                rprint(f"[red]La palabra '{query}' no está en el corpus ni cumple el formato híbrido.[/]")
                rprint(f"[yellow]¿Quisiste decir?[/]")
                for suggestion in suggestions:
                    rprint(f"  - {suggestion}")
                    continue
                else:
                    # Si no hay sugerencias, verificamos si es un verbo híbrido.
                    hybrid_vb = transcribir_hibrido_verbo_derivado_en_es(query)
            
                    if hybrid_vb:
                        rprint(f"[bold white]{query}[/] | [cyan]{hybrid_vb}[/] [italic](Análisis Híbrido)[/]")
                    else:
                        #No hay sugerencias ni es híbrida, marcamos el error.
                        rprint(f"[red]La palabra '{query}' no está en el corpus ni cumple el formato híbrido.[/]")
                
    lang = input("lang>> ").strip()
    if lang:
        rprint(f"Selected language: [yellow]{lang_codes.get(lang, lang)}[/]")
    else:
        rprint("Adios 👋🏼")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                       Representación fonética de palabras                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

NameError: name 'get_formated_string' is not defined


### Morfología

2. Elige tres lenguas del corpus que pertenezcan a familias lingüísticas distintas
   - Ejemplo: `spa` (Romance), `eng` (Germánica), `hun` (Urálica)
   - Para cada una de las tres lenguas calcula y compara:
       -  **Ratio morfemas / palabra**: El promedio de morfemas que componen las palabras
        -  **Indicé de Flexión / Derivación**: Del total de morfemas, ¿Qué porcentaje son etiquetas de flexión (`100`) y cuáles de derivación (`010`)?
3. Visualización
    - Genera una figura con **subplots** para comparar las lenguas lado a lado.
    - *Plot 1*: Distribución de la longitud de los morfemas
    - *Plot 2*: Distribución de las categorías (flexión, derivación, raíz, etc.)
4. Con base en esta información, responde la pregunta: *¿Cuál de las tres lenguas se comporta más como una lengua aglutinante y cuál como una lengua aislante?*
    - Justifica tu respuesta usando tus métricas y figuras



### EXTRA:

- Genera la [matriz de confusión](https://en.wikipedia.org/wiki/Confusion_matrix) para el etiquetador CRFs visto en clase
- Observando las etiquetas donde el modelo falló responde las preguntas:
    - ¿Por qué crees que se confundió?
    - ¿Es un problema de ambigüedad léxica (la palabra tiene múltiples etiquetas)?
    - ¿Qué *features* añadirías para solucionarlo?

|